In [1]:
import os
from pyspark.sql import SparkSession

spark_jars = os.environ.get("SPARK_JARS", "")
jar_list = spark_jars.split(",") if spark_jars else []
s3a_endpoint = os.environ.get("S3A_ENDPOINT", "")
s3a_access_key = os.environ.get("S3A_ACCESS_KEY", "")
s3a_secret_key = os.environ.get("S3A_SECRET_KEY", "")
driver_memory = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .appName("Tazama-Dashboard")
    .master("local[*]")
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", s3a_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory", driver_memory)
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128mb")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Version: {spark.version}")

26/05/20 08:13:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 08:13:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/20 08:13:26 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/05/20 08:13:26 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/05/20 08:13:26 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/05/20 08:13:26 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/05/20 08:13:26 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/05/20 08:13:26 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.


Spark Version: 3.4.2


In [2]:
WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")

Change the input path in the options parameters to create a view of any table in any layer, then you will be able to query via SparkSQL

In [11]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW hudi_table
USING hudi
OPTIONS (
  path '/opt/Tazama_Warehouse/gold/evaluation'
)
""")

DataFrame[]

In [12]:
result = spark.sql("""
SELECT *
FROM hudi_table
""")

In [13]:
result.show()

+-------------------+--------------------+--------------------+----------------------+--------------------+--------------------+--------------------+---------+-------------+--------+--------------------+-------------+-------------+---------+--------+---------------+----------------+--------------+--------------------+--------------------+--------------------+------------+------------+------------+--------------------+--------------------+-------------+-------------+---------------+--------------------+--------------------+---------+-------------------+--------------------+--------------------+---------------+---------------+---------------+------------------+--------------------+--------------------+--------------------+----------+
|_hoodie_commit_time|_hoodie_commit_seqno|  _hoodie_record_key|_hoodie_partition_path|   _hoodie_file_name|       evaluation_id|          message_id|tenant_id|report_status|is_alert|            event_ts|prcg_tm_dp_ns|prcg_tm_ed_ns|  tadp_id|tadp_cfg|tadp_prc